In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import matplotlib.pyplot as plt
import torch.nn.functional as F

# -- ⚙️ 1. 설정 (사용자 수정 영역) ---

# 테스트할 이미지가 담긴 폴더 경로

image_folder_path = r'C:\GITHUB\gardendoctor_ai-server\test_data\tomato'

# 저장된 모델 가중치 파일 경로

model_path = r'C:\GITHUB\gardendoctor_ai-server\ai_models\new_best_plant_model.pth'

# 학습 시 사용했던 클래스 이름 목록

class_names = ['고추', '딸기', '토마토']
# class_names = ['고추', '고추잎', '딸기', '딸기잎', '토마토', '토마토잎']

num_classes = len(class_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- 🧠 2. 모델 로드 ---

model = models.efficientnet_v2_m(weights=None)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
print("✅ 모델이 성공적으로 로드되었습니다.")

# -- 🖼️ 3. 이미지 전처리 정의 ---

inference_transform = transforms.Compose([
transforms.Resize(256),
transforms.CenterCrop(224),
transforms.ToTensor(),
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -- 📂 4. 폴더 내 이미지 추론 및 결과 시각화 ---

# 한글 폰트 설정

try:
    plt.rc('font', family='Malgun Gothic')
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print(f"폰트 설정 중 오류: {e}. 결과가 영문으로 표시될 수 있습니다.")

if not os.path.exists(image_folder_path):
    print(f"오류: '{image_folder_path}' 폴더를 찾을 수 없습니다.")
else:
    image_files = [f for f in os.listdir(image_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]


for image_file in image_files:
    image_path = os.path.join(image_folder_path, image_file)
    image = Image.open(image_path).convert('RGB')
    image_tensor = inference_transform(image)
    input_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)

        # [수정] 예측된 클래스 이름 가져오기
        predicted_class_string = class_names[predicted_idx.item()]
        confidence_percent = confidence.item() * 100

        # [수정] 클래스 이름을 '작물'과 '병명/상태'로 분리
        try:
            crop_name, disease_name = predicted_class_string.split('___')
            # 가독성을 위해 '_'를 공백으로 변경
            crop_name = crop_name.replace('_', ' ')
            disease_name = disease_name.replace('_', ' ')
        except ValueError:
            # '___'가 없는 경우를 대비한 예외 처리
            crop_name = predicted_class_string
            disease_name = "N/A"

        # [수정] 시각화 제목 형식 변경
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        title_text = (f"작물: {crop_name}\n"
                    f"진단명: {disease_name}\n"
                    f"신뢰도: {confidence_percent:.2f}%")
        plt.title(title_text, fontsize=16, loc='left', pad=15)
        plt.axis('off')
        plt.show()

# -- 📊 5. 정답률 계산 추가 ---

correct = 0
total = 0

for image_file in image_files:
    image_path = os.path.join(image_folder_path, image_file)
    image = Image.open(image_path).convert('RGB')
    image_tensor = inference_transform(image)
    input_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)

        predicted_class_string = class_names[predicted_idx.item()]
        
        # 예측이 뭔지 확인
        if predicted_class_string == '토마토' or predicted_class_string == '토마토잎':
            correct += 1
        total += 1

# 정답률 계산 및 출력
if total > 0:
    accuracy = (correct / total) * 100
    print(f"\n📈 총 이미지 수: {total}장")
    print(f"✅ '토마토'로 정확히 예측한 수: {correct}장")
    print(f"🎯 정답률: {accuracy:.2f}%")
else:
    print("⚠️ 처리할 이미지가 없습니다.")

In [ ]:
# -- 필요한 라이브러리 일괄 import ---

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import matplotlib.pyplot as plt
import torch.nn.functional as F

# -- ⚙️ 1. 설정 (사용자 수정 영역) ---

# [수정!] 테스트할 이미지가 담긴 폴더 경로

image_folder_path = r'C:\GITHUB\gardendoctor_ai-server\test_data\tomato'

# [수정!] 학습 코드에서 저장된 모델 가중치 파일 경로

model_path = r'C:\GITHUB\gardendoctor_ai-server\ai_models\best_convnext_model.pth'

# [!! 중요 !!] 학습 시 사용했던 클래스 이름 목록을 정확히 기입해야 합니다.

class_names = ['고추', '고추잎', '딸기', '딸기잎', '토마토', '토마토잎']

# -- 시스템 설정 (수정 불필요) ---

num_classes = len(class_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- 🧠 2. 모델 로드 ---

print("🧠 모델 로드를 시작합니다...")

model = models.convnext_base(weights=None)
num_ftrs = model.classifier[2].in_features
model.classifier[2] = nn.Sequential(
nn.Dropout(p=0.5),
nn.Linear(num_ftrs, num_classes)
)

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
print(f"✅ 모델이 성공적으로 로드되었습니다. (Device: {device})")

# -- 🖼️ 3. 이미지 전처리 정의 ---

# [TTA] TTA를 위한 변환 정의

# FiveCrop은 5개의 잘린 이미지를 포함한 튜플을 반환합니다.

tta_transform = transforms.Compose([
transforms.Resize(256),
transforms.FiveCrop(224) # 중앙, 좌상, 우상, 좌하, 우하 5개 영역을 자름
])

# 텐서 변환 및 정규화는 각 crop 이미지에 개별적으로 적용

to_tensor_transform = transforms.ToTensor()
normalize_transform = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# -- 📂 4. 폴더 내 이미지 추론 및 결과 시각화 ---

print(f"\n📂 '{image_folder_path}' 폴더의 이미지에 대한 TTA 추론을 시작합니다...")

# 한글 폰트 설정

try:
    plt.rc('font', family='Malgun Gothic')
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print(f"폰트 설정 중 오류: {e}.")

if not os.path.exists(image_folder_path):
    print(f"❌ 오류: '{image_folder_path}' 폴더를 찾을 수 없습니다.")
else:
    image_files = [f for f in os.listdir(image_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]


if not image_files:
    print(f"⚠️ 해당 폴더에 이미지 파일이 없습니다.")

for image_file in image_files:
    image_path = os.path.join(image_folder_path, image_file)

    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"이미지 파일 열기 실패: {image_path}, 오류: {e}")
        continue

    # [TTA] 10가지 버전의 이미지 생성 (5 crops * 2 flips)
    five_cropped_images = tta_transform(image)
    augmented_images = []
    for crop in five_cropped_images:
        # 원본 crop과 좌우 반전(hflip) crop을 모두 추가
        augmented_images.append(crop)
        augmented_images.append(transforms.functional.hflip(crop))

    # [TTA] 10개 버전을 하나의 배치로 만들어 정규화
    # ToTensorV2와 Normalize를 각 이미지에 적용 후 스택
    tta_batch = torch.stack([normalize_transform(to_tensor_transform(img)) for img in augmented_images]).to(device)

    with torch.no_grad():
        # [TTA] 10개 버전에 대해 한 번에 예측 수행
        outputs = model(tta_batch)
        probabilities = F.softmax(outputs, dim=1)

        # [TTA] 10개 버전의 예측 확률을 평균
        mean_probabilities = torch.mean(probabilities, dim=0)

        # [TTA] 평균 확률에서 최종 예측 결과 도출
        confidence, predicted_idx = torch.max(mean_probabilities, 0)

        predicted_class = class_names[predicted_idx.item()]
        confidence_percent = confidence.item() * 100

        # 결과 시각화
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        title_text = (f"[TTA 적용 결과]\n"
                    f"예측: {predicted_class}\n"
                    f"신뢰도: {confidence_percent:.2f}%")
        plt.title(title_text, fontsize=16, loc='left', pad=10)
        plt.axis('off')
        plt.show()
        
# -- 📊 5. 정답률 계산 추가 ---

correct = 0
total = 0

for image_file in image_files:
    image_path = os.path.join(image_folder_path, image_file)
    image = Image.open(image_path).convert('RGB')
    image_tensor = inference_transform(image)
    input_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)

        predicted_class_string = class_names[predicted_idx.item()]
        
        # 예측이 뭔지 확인
        if predicted_class_string == '토마토' or predicted_class_string == '토마토잎':
            correct += 1
        total += 1

# 정답률 계산 및 출력
if total > 0:
    accuracy = (correct / total) * 100
    print(f"\n📈 총 이미지 수: {total}장")
    print(f"✅ '토마토'로 정확히 예측한 수: {correct}장")
    print(f"🎯 정답률: {accuracy:.2f}%")
else:
    print("⚠️ 처리할 이미지가 없습니다.")

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import matplotlib.pyplot as plt
import torch.nn.functional as F
from tqdm import tqdm

# --- ⚙️ 1. 설정 (사용자 수정 영역) ---

# 테스트할 이미지가 담긴 폴더 경로
image_folder_path = r'C:\GITHUB\gardendoctor_ai-server\test_data\strawberry'

# 저장된 모델 가중치 파일 경로
model_path = r'C:\GITHUB\gardendoctor_ai-server\ai_models\new_finetuned_plant_model.pth'

# 학습 시 사용했던 클래스 이름 목록
class_names = ['고추', '딸기', '토마토']

# --- 시스템 설정 (수정 불필요) ---
num_classes = len(class_names)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 🧠 2. 모델 로드 ---
print("🧠 모델 로드를 시작합니다...")

# [수정] 모델 아키텍처를 EfficientNetV2-M으로 변경
model = models.efficientnet_v2_m(weights=None)
# [수정] EfficientNetV2-M에 맞는 분류기 구조
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)

# 저장된 가중치(state_dict) 불러오기
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval() # 모델을 반드시 평가 모드로 설정
print(f"✅ 모델이 성공적으로 로드되었습니다. (Device: {device})")


# --- 🖼️ 3. 이미지 전처리 정의 ---

# TTA를 위한 변환 정의
tta_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.FiveCrop(224) # 중앙, 좌상, 우상, 좌하, 우하 5개 영역을 자름
])
# 텐서 변환 및 정규화는 각 crop 이미지에 개별적으로 적용
to_tensor_transform = transforms.ToTensor()
normalize_transform = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


# --- 📂 4. 이미지 추론, 시각화, 정답률 계산 (루프 통합) ---
print(f"\n📂 '{image_folder_path}' 폴더의 이미지에 대한 TTA 추론을 시작합니다...")

# 한글 폰트 설정
try:
    plt.rc('font', family='Malgun Gothic')
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print(f"폰트 설정 중 오류: {e}.")

if not os.path.exists(image_folder_path):
    print(f"❌ 오류: '{image_folder_path}' 폴더를 찾을 수 없습니다.")
else:
    image_files = [f for f in os.listdir(image_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print("⚠️ 해당 폴더에 이미지 파일이 없습니다.")
    else:
        correct = 0
        total = 0
        
        for image_file in tqdm(image_files, desc="이미지 처리 중"):
            image_path = os.path.join(image_folder_path, image_file)
            
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"이미지 파일 열기 실패: {image_path}, 오류: {e}")
                continue

            # TTA: 10가지 버전의 이미지 생성 (5 crops * 2 flips)
            five_cropped_images = tta_transform(image)
            augmented_images = []
            for crop in five_cropped_images:
                augmented_images.append(crop)
                augmented_images.append(transforms.functional.hflip(crop))

            # TTA: 10개 버전을 하나의 배치로 만들어 정규화
            tta_batch = torch.stack([normalize_transform(to_tensor_transform(img)) for img in augmented_images]).to(device)
            
            with torch.no_grad():
                # TTA: 10개 버전에 대해 한 번에 예측 수행
                outputs = model(tta_batch)
                probabilities = F.softmax(outputs, dim=1)
                
                # TTA: 10개 버전의 예측 확률을 평균
                mean_probabilities = torch.mean(probabilities, dim=0)
                
                # TTA: 평균 확률에서 최종 예측 결과 도출
                confidence, predicted_idx = torch.max(mean_probabilities, 0)
                
                predicted_class = class_names[predicted_idx.item()]
                confidence_percent = confidence.item() * 100

            # 정답률 계산 로직
            if predicted_class == '딸기' or predicted_class == '딸기잎':
                correct += 1
            total += 1

            # 시각화 로직
            plt.figure(figsize=(8, 8))
            plt.imshow(image)
            title_text = (f"[TTA 적용 결과]\n"
                        f"예측: {predicted_class}\n"
                        f"신뢰도: {confidence_percent:.2f}%")
            plt.title(title_text, fontsize=16, loc='left', pad=10)
            plt.axis('off')
            plt.show()

        # --- 📊 5. 최종 정답률 출력 ---
        if total > 0:
            accuracy = (correct / total) * 100
            print(f"\n📈 총 이미지 수: {total}장")
            print(f"✅ '딸기'(으)로 정확히 예측한 수: {correct}장")
            print(f"🎯 정답률: {accuracy:.2f}%")
        else:
            print("⚠️ 처리된 이미지가 없습니다.")